# ETL: Precios de Criptomonedas (API → SQLite)

**Contexto de negocio:** una empresa de inversión quiere monitorear el mercado de criptomonedas. Te piden construir un pipeline que extraiga precios actuales desde una API pública, los transforme/limpie, y los cargue en una base de datos local para que el equipo pueda consultarlos con SQL.

Esto es un flujo **ETL** clásico:
- **Extract (Extraer):** llamar a la API y traer los datos crudos
- **Transform (Transformar):** limpiar, seleccionar columnas, dar formato
- **Load (Cargar):** guardar en una base de datos (SQLite) lista para consultar

**API que vamos a usar:** [CoinGecko](https://www.coingecko.com/en/api/documentation) — es gratuita, no necesita registro ni API key, y tiene límite generoso de peticiones para uso personal.

**Antes de empezar, instala lo que falte:**
```
pip install requests pandas
```
(`sqlite3` ya viene incluido en Python, no necesitas instalarlo.)

## Parte 1: EXTRACT — Llamar a la API

El endpoint que vamos a usar devuelve las top criptomonedas por capitalización de mercado:

```
https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&order=market_cap_desc&per_page=50&page=1
```

a) Usa la librería `requests` para hacer un `GET` a esa URL.

b) Revisa el `status_code` de la respuesta — si es `200`, salió bien. Si no, imprime el código para saber qué pasó.

c) Convierte la respuesta a JSON con `.json()` y guárdala en una variable `datos_crudos`.

d) Imprime cuántos elementos trajo (pista: `len(datos_crudos)`) y observa cómo se ve el primer elemento.

In [1]:
import requests

url = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&order=market_cap_desc&per_page=50&page=1"

# a) Hacemos la peticion GET: esto es "tocar la puerta" de la API y pedirle los datos
respuesta = requests.get(url)

In [2]:
# b)El status_code nos dice como salio la peticion: 200 = exito, 404 = no encontrado, 429 = demasiadas peticiones, etc.
print("Status code:", respuesta.status_code)

Status code: 200


In [3]:
# c) La respuesta de la API viene como texto en formato JSON.
# .json() lo convierte automaticamente en una lista de diccionarios de Python, lista para usar
datos_crudos = respuesta.json()

In [4]:
# d) Contamos cuantos elementos (monedas) vinieron en la respuesta
print("Cantidad de monedas:", len(datos_crudos))

# Mostramos el primer elemento para ver la estructura completa de una moneda
# (al ser la ultima linea de la celda, Jupyter lo muestra automaticamente sin necesitar print)
datos_crudos[0]

Cantidad de monedas: 50


{'id': 'bitcoin',
 'symbol': 'btc',
 'name': 'Bitcoin',
 'image': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400',
 'current_price': 79902,
 'market_cap': 1604423508745,
 'market_cap_rank': 1,
 'fully_diluted_valuation': 1604429021909,
 'total_volume': 18809541440,
 'high_24h': 80147,
 'low_24h': 79459,
 'price_change_24h': 245.19,
 'price_change_percentage_24h': 0.29614,
 'market_cap_change_24h': 4937561931,
 'market_cap_change_percentage_24h': 0.3087,
 'circulating_supply': 20080162.0,
 'total_supply': 20080256.0,
 'max_supply': 21000000.0,
 'ath': 126080,
 'ath_change_percentage': -36.62586,
 'ath_date': '2025-10-06T10:57:42.000Z',
 'atl': 67.81,
 'atl_change_percentage': 117733.98203,
 'atl_date': '2013-07-05T16:00:00.000Z',
 'roi': None,
 'last_updated': '2026-09-06T00:38:20.000Z'}

## Parte 2: TRANSFORM — Limpiar y estructurar

a) Convierte `datos_crudos` (una lista de diccionarios) en un DataFrame de pandas con `pd.DataFrame()`.

b) Explora las columnas disponibles con `.columns` — vas a ver que la API trae mucho más de lo que necesitas.

c) Quédate solo con las columnas relevantes para el negocio:
```python
columnas = ['id', 'symbol', 'name', 'current_price', 'market_cap',
            'market_cap_rank', 'total_volume', 'high_24h', 'low_24h',
            'price_change_percentage_24h']
```

d) Renombra las columnas a español para que sea más claro para el equipo de negocio (pista: `.rename(columns={...})`). Ejemplo: `current_price` → `precio_actual_usd`.

e) Agrega una columna `fecha_extraccion` con la fecha y hora en que corriste el script (pista: `pd.Timestamp.now()`). Esto es clave en cualquier ETL — siempre necesitas saber CUÁNDO se extrajeron los datos.

f) Revisa si hay nulos con `.isna().sum()` y decide qué hacer si los hay.

In [9]:
import pandas as pd

# a) Convertimos la lista de diccionarios en una tabla (DataFrame) de pandas
df = pd.DataFrame(datos_crudos)

# b) Vemos todas las columnas disponibles -- son muchas mas de las que necesitamos
print(df.columns.tolist())

['id', 'symbol', 'name', 'image', 'current_price', 'market_cap', 'market_cap_rank', 'fully_diluted_valuation', 'total_volume', 'high_24h', 'low_24h', 'price_change_24h', 'price_change_percentage_24h', 'market_cap_change_24h', 'market_cap_change_percentage_24h', 'circulating_supply', 'total_supply', 'max_supply', 'ath', 'ath_change_percentage', 'ath_date', 'atl', 'atl_change_percentage', 'atl_date', 'roi', 'last_updated']


In [ ]:
# c) Definimos que columnas si nos sirven (descartamos imagen, roi, fechas de maximo/minimo historico, etc.)
columnas = ['id', 'symbol', 'name', 'current_price', 'market_cap',
            'market_cap_rank', 'total_volume', 'high_24h', 'low_24h',
            'price_change_percentage_24h']

# Filtramos el DataFrame para que solo tenga esas columnas
df = df[columnas]

df.head()

,id,symbol,name,current_price,market_cap,market_cap_rank,total_volume,high_24h,low_24h,price_change_percentage_24h
0,bitcoin,btc,Bitcoin,79902.00,1604423508745,1,1.880954e+10,80147.00,79459.000000,0.29614
1,ethereum,eth,Ethereum,2488.23,303596441259,2,6.824871e+09,2490.78,2445.890000,1.32728
2,tether,usdt,Tether,1.00,183404718521,3,3.904804e+10,1.00,0.999885,-0.01825
3,binancecoin,bnb,BNB,768.05,102288952719,4,2.202394e+09,779.70,719.580000,6.54796
4,ripple,xrp,XRP,1.42,88854787408,5,1.475214e+09,1.43,1.390000,1.32313


In [11]:
# d) Renombramos las columnas para que cualquiera del equipo de negocio las entienda sin tener que traducir
df = df.rename(columns={
    'id': 'id_moneda',
    'symbol': 'simbolo',
    'name': 'nombre',
    'current_price': 'precio_actual_usd',
    'market_cap': 'capitalizacion_mercado',
    'market_cap_rank': 'ranking',
    'total_volume': 'volumen_24h',
    'high_24h': 'maximo_24h',
    'low_24h': 'minimo_24h',
    'price_change_percentage_24h': 'cambio_porcentual_24h'
})

df.head()

,id_moneda,simbolo,nombre,precio_actual_usd,capitalizacion_mercado,ranking,volumen_24h,maximo_24h,minimo_24h,cambio_porcentual_24h
0,bitcoin,btc,Bitcoin,79902.00,1604423508745,1,1.880954e+10,80147.00,79459.000000,0.29614
1,ethereum,eth,Ethereum,2488.23,303596441259,2,6.824871e+09,2490.78,2445.890000,1.32728
2,tether,usdt,Tether,1.00,183404718521,3,3.904804e+10,1.00,0.999885,-0.01825
3,binancecoin,bnb,BNB,768.05,102288952719,4,2.202394e+09,779.70,719.580000,6.54796
4,ripple,xrp,XRP,1.42,88854787408,5,1.475214e+09,1.43,1.390000,1.32313


In [13]:
# e) Registramos la fecha y hora exactas en que extrajimos estos datos.
# Es clave en cualquier ETL: los precios cambian, y sin esto no sabrias
# si estos numeros son de hoy o de hace una semana
df['fecha_extraccion'] = pd.Timestamp.now()

df.head()

,id_moneda,simbolo,nombre,precio_actual_usd,capitalizacion_mercado,ranking,volumen_24h,maximo_24h,minimo_24h,cambio_porcentual_24h,fecha_extraccion
0,bitcoin,btc,Bitcoin,79902.00,1604423508745,1,1.880954e+10,80147.00,79459.000000,0.29614,2026-09-05 19:00:40.803885
1,ethereum,eth,Ethereum,2488.23,303596441259,2,6.824871e+09,2490.78,2445.890000,1.32728,2026-09-05 19:00:40.803885
2,tether,usdt,Tether,1.00,183404718521,3,3.904804e+10,1.00,0.999885,-0.01825,2026-09-05 19:00:40.803885
3,binancecoin,bnb,BNB,768.05,102288952719,4,2.202394e+09,779.70,719.580000,6.54796,2026-09-05 19:00:40.803885
4,ripple,xrp,XRP,1.42,88854787408,5,1.475214e+09,1.43,1.390000,1.32313,2026-09-05 19:00:40.803885


In [ ]:
# f) Verificamos si alguna de las columnas que nos quedamos tiene valores faltantes
print(df.isna().sum())

id_moneda                 0
simbolo                   0
nombre                    0
precio_actual_usd         0
capitalizacion_mercado    0
ranking                   0
volumen_24h               0
maximo_24h                0
minimo_24h                0
cambio_porcentual_24h     0
fecha_extraccion          0
dtype: int64


## Parte 3: LOAD — Cargar a SQLite

a) Crea (o conéctate a) una base de datos SQLite en `../data/cripto.db` usando el módulo `sqlite3`.

b) Usa `df.to_sql()` para guardar tu DataFrame como una tabla llamada `precios_cripto`. Usa `if_exists='append'` (no `'replace'`) — así, cada vez que corras el notebook, vas acumulando un histórico de precios en el tiempo, en vez de borrar lo anterior.

c) Cierra la conexión con `.close()` (o usa un bloque `with` para que se cierre solo).

In [6]:
import sqlite3

# Tu código aquí


## Parte 4: Verifica con SQL

Ahora que los datos están en SQLite, practica consultarlos con SQL directamente (no con pandas) — es una habilidad que todo analista necesita.

a) Conéctate a la base y usa `pd.read_sql("SELECT * FROM precios_cripto LIMIT 10", conexion)` para ver una muestra.

b) Escribe una consulta SQL que traiga las 5 criptomonedas con mayor `precio_actual_usd`.

c) Escribe una consulta SQL que cuente cuántos registros hay en total en la tabla (útil para confirmar que se está acumulando histórico si corres el notebook varias veces en días distintos).

In [7]:
# Tu código aquí


## Parte 5: Preguntas de negocio

Con los datos ya en tu DataFrame (o consultando la base):

a) ¿Qué criptomoneda tuvo el mayor cambio porcentual positivo en las últimas 24h? ¿Y el mayor cambio negativo?

b) ¿Cuál es el rango de precios (`high_24h` - `low_24h`) más amplio, en términos absolutos? Eso indica cuál fue la más volátil del día.

c) Suma el `market_cap` de las top 10 monedas. ¿Qué porcentaje del total de las 50 que trajiste representan esas top 10? (esto te da una idea de qué tan concentrado está el mercado).

In [8]:
# Tu código aquí


## Parte 6 (Bonus): Automatiza el pipeline

Este es el paso que convierte tu notebook en un proyecto de portafolio realmente sólido: un ETL de verdad no se corre a mano cada vez, se automatiza.

Ve al archivo `../scripts/etl_cripto.py` — ahí tienes el mismo pipeline (Extract, Transform, Load) pero como script ejecutable, listo para programarse con el Programador de Tareas de Windows y correr, por ejemplo, una vez al día. Así, con el tiempo, vas acumulando un histórico real de precios en tu base de datos — información que después podrías graficar como serie de tiempo.

No necesitas hacer nada más aquí — solo revisa ese script y, si quieres, intenta correrlo desde la terminal con `python scripts/etl_cripto.py` para confirmar que funciona igual que tu notebook.

## Conclusiones (para tu README)

Escribe aquí 3-5 líneas resumiendo: qué construiste, qué retos encontraste (límites de la API, formato de datos, etc.) y qué aprendiste sobre trabajar con fuentes de datos externas en vez de archivos estáticos.

*(Tus conclusiones aquí)*